# Exercise 3 — Validate the sales

**Learner exercise** · [All exercises](../index.html) · [Setup](../README.md)

## What you’ll learn

- Interpret parsed values and rejection reasons from the supplied cleaning step.
- Separate accepted and rejected sales and account for every input row.

**Core: about 5 minutes.** The same baseline for everyone. [Optional zoom-in](#zoom): about 6 extra minutes; choose it here if the topic interests you.

Complete **Your code**, run the **Check** cells, and open hints when needed. Replace `todo(...)` with your answer. Do not use **Run All** while tasks remain unfinished.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../RECOVERY.md).

In [ ]:
from pathlib import Path
import sys

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'workshop_runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

import lab_checks as check
from arrival_files import publish_arrival
from lab_checks import todo
from lab_workspace import Workspace
from workshop_runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path

workspace = Workspace(solutions=False)
product_key, clean_products = workspace.load('product_key', 'clean_products')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

---
<a id="exercise-3"></a>
## Your task

**Which rows belong in the report, and how will we explain the rejects?**

Core budget: about 5 minutes.

Our contract rejects missing/empty product keys, invalid or missing amounts, and invalid or missing timestamps. Keep the raw values and a `reject_reason`. A valid key absent from the lookup is still an accepted sale.

Parse amounts as `DECIMAL(12, 2)` and timestamps with `yyyy-MM-dd HH:mm:ss`. Parsing failures should become null, so we can explain them. Do not silently turn missing amounts into zero.

### Predict before running

Which sale IDs will fail this contract? Does a missing product lookup make a sale invalid?

Your prediction: …

### Supplied — parse and mark bad rows

This parsing function is supplied for the core. Read its rules before using it; the optional section lets you build the parsing expressions yourself.

In [ ]:
def clean_sales(raw: DataFrame) -> DataFrame:
    """Parse the exercise's UTC timestamps and amounts; retain bad input."""
    return (
        raw.withColumn("product_id", product_key(F.col("product_id")))
        .withColumn("amount", F.expr("try_cast(amount_raw AS DECIMAL(12, 2))"))
        .withColumn(
            "sold_at",
            F.try_to_timestamp("sold_at_raw", F.lit("yyyy-MM-dd HH:mm:ss")),
        )
        .withColumn(
            "reject_reason",
            F.when(
                F.col("product_id").isNull() | (F.col("product_id") == ""),
                F.lit("missing product key"),
            )
            .when(F.col("amount").isNull(), F.lit("invalid or missing amount"))
            .when(F.col("sold_at").isNull(), F.lit("invalid or missing timestamp")),
        )
    )

In [ ]:
cleaned = clean_sales(raw)
cleaned.select("sale_id", "product_id", "amount", "sold_at", "reject_reason").orderBy(
    "sale_id"
).show(truncate=False)

### Your code — separate accepted and rejected rows

Keep the filtering functions reusable: the stream will call them too.

In [ ]:
def accepted_sales(cleaned: DataFrame) -> DataFrame:
    """Return sales without a rejection reason."""
    return cleaned.filter(todo("3: select rows with no reject_reason"))


def rejected_sales(cleaned: DataFrame) -> DataFrame:
    """Return rejected sales, keeping their raw values and reason."""
    return cleaned.filter(todo("3: select rows with a reject_reason"))

In [ ]:
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
rejected.select("sale_id", "amount_raw", "sold_at_raw", "reject_reason").orderBy("sale_id").show(
    truncate=False
)

### Check

Reconcile all eight input rows. Check which rows survive, not just how many.

In [ ]:
check.validation(raw, accepted, rejected)

<details>
<summary>Need a nudge? Hint 1</summary>

Use the tolerant parsing functions named in the task. Test parsed values for null.

</details>

<details>
<summary>A little more help: Hint 2</summary>

A chained `when` without an `otherwise` produces null when none of its conditions matches. Filter that column with `isNull()` or `isNotNull()`.

</details>

If you need to catch up during class, use the explicit [recovery step](../RECOVERY.md#exercise-3). [Worked solution](../solutions/03-validate.ipynb) — open it separately when you are ready to compare.

## Core complete

For the 60-minute lab, [skip to Save and finish](#finish). To explore this topic further, continue with the optional section below. Later core exercises do not need any of its variables.

<a id="zoom"></a>
## Optional zoom-in · about 6 minutes

These investigations make up the extra depth in a 90-minute session. Choose them independently; keep your working pipeline unchanged.

### Parseable does not always mean valid

Our contract is intentionally small. It does not, for example, reject a negative amount: refunds might be legitimate. A real pipeline needs an explicit business decision about that.

Keep `amount_raw` and `sold_at_raw` alongside parsed columns so a reject can be explained. Spark null checks use `isNull()`/`isNotNull()`, not Python's `is None`. The [parsing experiment](deeper/schemas-and-parsing.ipynb) adds malformed and missing keys/timestamps without changing the main fixture.

### Your code — build the parsing expressions

Create a separate `parsed_preview` with `sale_id`, parsed `amount` as `DECIMAL(12, 2)`, and parsed `sold_at` using `yyyy-MM-dd HH:mm:ss`. Use tolerant parsing so invalid input becomes null. Compare it with the supplied cleaner without changing that function.

In [ ]:
parsed_preview = raw.select(
    "sale_id",
    todo("Parse amount_raw as a decimal without failing on invalid values").alias("amount"),
    todo("Parse sold_at_raw with the given timestamp pattern").alias("sold_at"),
)

In [ ]:
check.same_rows(cleaned.select("sale_id", "amount", "sold_at"), parsed_preview)

<details><summary>Hint for the parsing expressions</summary>

Use `F.expr` with SQL `try_cast` for the decimal and `F.try_to_timestamp` with `F.lit` for the timestamp pattern. Both expressions should preserve a row even when parsing fails.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [ ]:
workspace.save(clean_sales, accepted_sales, rejected_sales)
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Next: [Exercise 4 — Join and aggregate](04-join-aggregate.ipynb).

Want more on this topic? You can open these now, using the same saved work: [Schemas and parsing](deeper/schemas-and-parsing.ipynb).

After your attempt, compare the separate [worked solution](../solutions/03-validate.ipynb).